# 批次規範化（Batch Normalization, BN）

是一種在深度神經網路中用來穩定與加速訓練的技術。

它的核心思想是：在每一層的輸入（或線性變換後、激活函數前的輸出）上，對每一個 mini-batch 做標準化處理，讓資料分佈維持在較穩定的範圍。

In [ ]:
from NeuralNetwork import *

class BatchNorm_1d(Layer):
    def __init__(self, num_features, gamma_beta_method=None, eps=1e-8, momentum=0.9):
       # 初始化函數：設定 Batch Normalization 層的參數
       # self.d_X, self.h_X, self.w_X = X_dim
       # self.gamma = np.ones((1, int(np.prod(X_dim)) ))
       # self.beta = np.zeros((1, int(np.prod(X_dim))))
       # self.params = [self.gamma,self.beta]
        super().__init__()
        self.eps = eps            # 避免除以零的小常數（微小值）
        self.momentum = momentum  # 動量參數，用來更新移動平均值（Running Mean/Var）
        
        # 根據 method 初始化權重 gamma 和偏置 beta
        if not gamma_beta_method:
            self.gamma = np.ones((1, num_features ))
            self.beta = np.zeros((1, num_features ))             
        else:
            self.gamma = np.random.randn(1, num_features)
            self.beta =  np.random.randn(1, num_features)  # 使用隨機常態分佈初始化 
         
        # 初始化移動平均的平均值與變異數，用於推論階段（Inference）
        self.running_mu = np.zeros((1, num_features ))  
        self.running_var = np.zeros((1, num_features )) 
        
        # 儲存可學習參數與其對應的梯度（Gradient）
        self.params = [self.gamma, self.beta]
        self.grads = [np.zeros_like(self.gamma), np.zeros_like(self.beta)]

    def forward(self, X, training=True):
        # 前向傳播（Forward Pass）
        if training: 
            self.n_X = X.shape[0]  # Batch size (樣本數量)
            self.X_shape = X.shape

            # 將輸入資料拉平，方便計算每一個特徵維度的統計量
            self.X_flat = X.ravel().reshape(self.n_X, -1)
            # 計算目前這個 Batch 的平均值與變異數
            self.mu = np.mean(self.X_flat, axis=0)
            self.var = np.var(self.X_flat, axis=0) 
            
            # 進行標準化（Normalization）
            self.X_hat = (self.X_flat - self.mu) / np.sqrt(self.var + self.eps)
            # 縮放與平移（Scale and Shift）
            out = self.gamma * self.X_hat + self.beta

            # 計算並更新平均值與變異數的移動平均（EMA）
            running_mu, running_var, momentum = self.running_mu, self.running_var, self.momentum
            running_mu = momentum * running_mu + (1 - momentum) * self.mu
            running_var = momentum * running_var + (1 - momentum) * self.var            
        else:             
            # 測試/推論階段：使用訓練時累積的移動平均值
            X_flat = X.ravel().reshape(X.shape[0], -1)
            # 規範化
            X_hat = (X_flat - running_mean) / np.sqrt(running_var + eps)
            # 縮放與平移
            out = self.gamma * X_hat + self.beta          
        return out.reshape(self.X_shape)

    
    def __call__(self, X):
        return self.forward(X)

    def backward(self, dout):
        # 反向傳播（Backward Pass）：計算損失函數對各參數的導數
        eps = self.eps
        # 將輸入的梯度張量調整為二維形狀
        dout = dout.ravel().reshape(dout.shape[0], -1)
        X_mu = self.X_flat - self.mu
        var_inv = 1. / np.sqrt(self.var + eps)
        
        # 計算對 beta 和 gamma 的梯度（用於更新參數）
        dbeta = np.sum(dout, axis=0)
        dgamma = np.sum(dout * self.X_hat, axis=0)

        # 鏈鎖律連鎖反應：計算對輸入 X 的梯度
        dX_hat = dout * self.gamma
        # 對變異數的偏微分
        dvar = np.sum(dX_hat * X_mu, axis=0) * -0.5 * (self.var + eps)**(-3/2)           
        # 對平均值的偏微分
        dmu = np.sum(dX_hat * (-var_inv), axis=0) + dvar * 1/self.n_X * np.sum(-2.* X_mu, axis=0)
        # 最終對輸入資料 X 的梯度
        dX = (dX_hat * var_inv) + (dmu / self.n_X) + (dvar * 2/self.n_X * X_mu)        
        dX = dX.reshape(self.X_shape)
        
        # 累加梯度到 grads 列表中，待優化器（Optimizer）更新
        self.grads[0] += dgamma
        self.grads[1] += dbeta
        return dX